# tAge mortality

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import hashlib
import inspect
import json
import os
import shutil
import urllib.request
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer

import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.TAge)
print_entire_class(pya.models.TAgeMortality)

class TAge(pyagingModel):
    def __init__(self):
        super().__init__()
        # Cohort-relative clock: input must come from ``preprocess.prepare_tage``.
        self.required_uns_flag = "tage_prepared"

    def preprocess(self, x):
        """Substitute the fitted imputer statistics for genes absent from the sample.

        ``reference_values`` holds the published pipeline's ``SimpleImputer`` fill
        values, so a gene the aligned list carries but the sample does not gets the
        exact value the model was fitted to expect. For a cohort-centered clock that
        is the least-biased choice available: the gene contributes its training mean
        rather than pulling the score toward zero or toward this cohort's center.
        """
        if self.reference_values is None:
            return x
        if isinstance(self.reference_values, torch.Tensor):
            reference = self.reference_values.to(device=x.device, dtype=x.dtype)
        else:
            reference = t

In [3]:
model = pya.models.TAgeMortality()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "tagemortality"
model.metadata["data_type"] = "transcriptomics (relative)"  # Paper: RNA-seq, reference-centred
model.metadata["species"] = "multiple species"  # Paper: mouse, rat, macaque and human via mouse orthologs
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Tyshkovskiy, Alexander, et al. \"Universal transcriptomic hallmarks of mammalian ageing and mortality.\" Nature 654 (2026): 173-188."
model.metadata["doi"] = "https://doi.org/10.1038/s41586-026-10542-3"
# MGB Open Access License 1.0 on the Zenodo record: non-commercial academic use only.
model.metadata["research_only"] = True
model.metadata["notes"] = "Elastic Net over 10,487 mouse-Entrez genes, multispecies multi-tissue, scaleddiff variant. Cohort-relative: inputs must come from pyaging.preprocess.prepare_tage, and a prediction is a hazard shift against the chosen reference group, not an absolute risk. The published pipeline's SimpleImputer, mean-only StandardScaler and pass-through SelectKBest are folded into the packaged linear layer, and the imputer medians are carried as reference_values so a gene the sample does not measure contributes its training median. Output is log10(hazard ratio) -- base 10, not the natural log the 'log hazard' unit label usually implies -- and unlike the chronological clocks it is never rescaled by species maximum lifespan, so it is directly comparable across species. Released under the MGB Open Access License 1.0: non-commercial academic research use only."
model.metadata["tissue"] = ["multi-tissue"]  # Paper: multi-tissue
model.metadata["predicts"] = ["mortality risk"]  # Paper: transcriptomic mortality risk
model.metadata["training_target"] = ["mortality"]  # Paper: mortality
model.metadata["unit"] = ["log hazard"]  # Paper: log10(hazard ratio); base 10, not natural log
model.metadata["model_type"] = "elastic net regression"  # Paper: ElasticNet, alpha=0.001, l1_ratio=0.2
model.metadata["platform"] = ["RNA-seq"]  # Paper: RNA-seq
model.metadata["population"] = "multiple mammalian species"  # Paper: mouse, rat, macaque, human
model.metadata["journal"] = "Nature"
model.metadata["last_author"] = "Vadim N. Gladyshev"
model.metadata["n_features"] = 10487
model.metadata["citations"] = 0
model.metadata["citations_date"] = "2026-08-21"

## Download clock dependencies

The published model is a fitted `sklearn.pipeline.Pipeline` on the paper's Zenodo record. It is
downloaded here rather than checked in, and the SHA-256 below is the one recorded in
`tests/data/tage/README.md`, so a silently changed upload fails loudly instead of quietly
re-deriving different weights.

In [5]:
FILENAME = "EN_Mortality_Multispecies_Multitissue_scaleddiff.pkl"
SHA256 = "458df7680cfa1422fb0aea42dcb1959142d55b9195e993e26bacee4807525566"
URL = f"https://zenodo.org/records/18763485/files/{FILENAME}?download=1"

if not os.path.exists(FILENAME):
    urllib.request.urlretrieve(URL, FILENAME)

digest = hashlib.sha256(Path(FILENAME).read_bytes()).hexdigest()
assert digest == SHA256, f"Zenodo download does not match the recorded checksum: {digest}"
digest

'458df7680cfa1422fb0aea42dcb1959142d55b9195e993e26bacee4807525566'

In [6]:
# joblib.load executes pickle bytecode. It is acceptable here because the bytes came from the
# paper's official Zenodo record over HTTPS at a fixed record id and were checksummed above --
# the same trust decision pyaging already makes for its own torch.save'd clocks.
#
# The pipeline was fitted under scikit-learn 1.3.2. Its SimpleImputer predates the _fill_dtype
# attribute newer versions expect, so it is restored from statistics_ exactly as the authors'
# own inst/python/tage_predict.py does. Unpickling also warns about the version skew; the
# predictions were verified to reproduce regardless (see tests/data/tage/README.md).
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline = joblib.load(FILENAME)

for _, step in pipeline.steps:
    if isinstance(step, SimpleImputer) and not hasattr(step, "_fill_dtype"):
        step._fill_dtype = step.statistics_.dtype

for step_name, step in pipeline.steps:
    print(f"{step_name:<16} {type(step).__name__:<16} n_features_in_={step.n_features_in_}")

# Pin the shape of the pipeline: an extra or reordered step would invalidate the collapse below,
# and it must fail here rather than produce plausible-looking wrong weights.
assert [step_name for step_name, _ in pipeline.steps] == [
    "imputation",
    "scaler",
    "featureselection",
    "estimator",
]
assert [type(step).__name__ for _, step in pipeline.steps] == [
    "SimpleImputer",
    "StandardScaler",
    "SelectKBest",
    "ElasticNet",
]
assert {step.n_features_in_ for _, step in pipeline.steps} == {pipeline.n_features_in_}

imputation       SimpleImputer    n_features_in_=10487
scaler           StandardScaler   n_features_in_=10487
featureselection SelectKBest      n_features_in_=10487
estimator        ElasticNet       n_features_in_=10487


### Resolving the four pipeline steps

Every step has to be reproduced or refuted as identity before the Elastic Net coefficients mean
anything on their own. Printed above and asserted below:

| Step | What it does | How pyaging reproduces it |
|---|---|---|
| `imputation` | `SimpleImputer(strategy="median")` over all 10 487 inputs | carried as `model.reference_values`; `TAge.preprocess` substitutes them for `NaN`, and `check_features_in_adata` uses the same values for genes the input lacks entirely |
| `scaler` | `StandardScaler(with_mean=True, with_std=False)` -- centring only, **no** variance scaling | folded into the packaged bias |
| `featureselection` | `SelectKBest(k=10487)` over 10 487 inputs -- selects everything | asserted to be a pass-through, so nothing to reproduce |
| `estimator` | `ElasticNet` | the packaged linear layer |

Because the last three steps are all affine, they collapse exactly into a single linear layer.
With the selector's support mask `S`, the scaler's `mean_` and `scale_`, and the Elastic Net's
`coef_` and `intercept_`:

```
w[j] = coef_[i] / scale_[j]           for the i-th selected input j, and 0 for unselected inputs
b    = intercept_ - sum_i coef_[i] * mean_[j_i] / scale_[j_i]
```

The general form is written out below even though `with_std=False` makes every `scale_[j]` equal
to 1 and the selector selects everything here: a hardcoded shortcut would silently produce wrong
weights if the file on Zenodo were ever refitted with different settings, whereas the general
form plus the assertions below stays correct or fails loudly.

## Load features

`feature_names_in_` is the authority on both the names and their order. They are mouse Entrez IDs
as strings; `prepare_tage` delivers columns in exactly that ID space.

In [7]:
features = [str(f) for f in pipeline.feature_names_in_]
assert len(features) == len(set(features)) == pipeline.n_features_in_
model.features = features
len(model.features), model.features[:5]

(10487, ['16870', '18707', '67388', '23888', '14275'])

## Load weights into base model

In [8]:
imputer = pipeline.named_steps["imputation"]
scaler = pipeline.named_steps["scaler"]
selector = pipeline.named_steps["featureselection"]
estimator = pipeline.named_steps["estimator"]

# A StandardScaler with with_std=False leaves scale_ as None, and with_mean=False leaves mean_ as
# None; substituting the identity values keeps the collapse below in its general form.
scale = scaler.scale_ if scaler.with_std else np.ones(scaler.n_features_in_)
mean = scaler.mean_ if scaler.with_mean else np.zeros(scaler.n_features_in_)

selected = np.flatnonzero(selector.get_support())
assert len(selected) == estimator.coef_.shape[0]
assert scale.shape == mean.shape == (len(features),)  # scaler stats are indexed pre-selection

weight = np.zeros(len(features), dtype=np.float64)
weight[selected] = estimator.coef_ / scale[selected]
bias = float(estimator.intercept_ - np.sum(estimator.coef_ * mean[selected] / scale[selected]))

int(np.count_nonzero(weight)), bias

(10487, -0.016780892841450207)

The mortality clock predicts log10(hazard ratio) directly and, unlike the chronological
clocks, is never rescaled by species maximum lifespan. The factor below is therefore 1, and it is
written out rather than dropped so this notebook and `tage.ipynb` stay diffable.

In [9]:
LIFESPAN_SCALE = 1.0  # mortality clocks are never rescaled by species maximum lifespan

model.base_model = pya.models.LinearModel(input_dim=len(model.features))
model.base_model.linear.weight.data = torch.tensor(
    weight * LIFESPAN_SCALE, dtype=torch.float64
).unsqueeze(0)
model.base_model.linear.bias.data = torch.tensor([bias * LIFESPAN_SCALE], dtype=torch.float64)

model.base_model.linear.weight.shape, model.base_model.linear.bias

(torch.Size([1, 10487]),
 Parameter containing:
 tensor([-0.0168], dtype=torch.float64, requires_grad=True))

## Load reference values

`reference_values` holds the fitted imputer's medians, one per feature, in feature order. Two
different gaps land on them: a gene the aligned matrix carries as `NaN` (padded by
`prepare_tage` because the experiment did not measure it) is substituted in `TAge.preprocess`,
and a gene absent from `adata.var_names` altogether is filled by `check_features_in_adata`.
Both reproduce what the published pipeline's own `SimpleImputer` would have done. Filling with
zero instead would be wrong here: zero is a meaningful value in centred space ("identical to the
reference group"), so it would assert a measurement that was never taken.

In [10]:
assert imputer.strategy == "median"
assert np.isnan(imputer.missing_values)
assert imputer.statistics_.shape == (len(model.features),)
assert not np.isnan(imputer.statistics_).any()

model.reference_values = [float(v) for v in imputer.statistics_]
len(model.reference_values), model.reference_values[:5]

(10487,
 [-0.0113398896172976, 0.0, 0.0, 0.00068916012986555, 0.0178664836280324])

## Load preprocess and postprocess objects

Both are `None`: the imputer substitution lives in `TAge.preprocess` as a property of the class,
not as a named transform with dependencies, and there is nothing left to do after the linear
layer -- the lifespan rescale is already inside the weights.

In [11]:
model.preprocess_name = None
model.preprocess_dependencies = None
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Tyshkovskiy, Alexander, et al. "Universal transcriptomic '
             'hallmarks of mammalian ageing and mortality." Nature 654 (2026): '
             '173-188.',
 'citations': 0,
 'citations_date': '2026-08-21',
 'clock_name': 'tagemortality',
 'data_type': 'transcriptomics (relative)',
 'doi': 'https://doi.org/10.1038/s41586-026-10542-3',
 'journal': 'Nature',
 'last_author': 'Vadim N. Gladyshev',
 'model_type': 'elastic net regression',
 'n_features': 10487,
 'notes': 'Elastic Net over 10,487 mouse-Entrez genes, multispecies '
          'multi-tissue, scaleddiff variant. Cohort-relative: inputs must come '
          'from pyaging.preprocess.prepare_tage, and a prediction is a hazard '
          'shift against the chosen reference group, not an absolute risk. The '
          "published pipeline's SimpleImput

## Normal feature ranges

Reference-centred expression has no plausible bound in either direction and no registry unit, so
every feature comes back unbounded with a `None` unit -- exactly the modality default for
`transcriptomics (relative)`.

In [13]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
assert all(record["low"] == -np.inf and record["high"] == np.inf for record in feature_ranges)
assert set(model.feature_units) == {None}
pd.DataFrame.from_records(feature_ranges).head()

,feature,unit,low,high
0,16870,None,-inf,inf
1,18707,None,-inf,inf
2,67388,None,-inf,inf
3,23888,None,-inf,inf
4,14275,None,-inf,inf


## Basic test

A row of zeros is the reference point itself: a sample whose every gene sits exactly at the
reference-group median. The prediction there is the clock's baseline, and it should be close to
zero -- a sample indistinguishable from the reference is neither older- nor younger-looking.

In [14]:
model.eval()
model.to(torch.float64)
with torch.inference_mode():
    baseline = model(torch.zeros((1, len(model.features)), dtype=torch.float64))
baseline.item()

-0.016780892841450207

#### Parity with the published sklearn pipeline

`tests/data/tage/expected_predictions.json` was produced by running the published pipeline itself
on the authors' example data (24 mouse kidney and skeletal-muscle samples), so agreeing with it
cannot be self-confirmation -- no pyaging code path contributed to those numbers. The stage
matrices are written in R orientation (genes as rows), so they are transposed first.

The aligned matrices still carry `NaN` for the 3 542 genes the gene list has but this dataset does
not, of which 375 are model features. Feeding them through untouched is the point: it is the
`TAge.preprocess` substitution, not a cleaned-up input, that has to reproduce the imputer.

In [15]:
FIXTURES = Path("../../tests/data/tage")
expected = json.loads((FIXTURES / "expected_predictions.json").read_text())


def load_stage(centering):
    """Return the fixture stage matrix as samples x model features, NaNs intact."""
    frame = pd.read_csv(FIXTURES / f"after_{centering}.csv.gz", index_col=0).T
    frame.columns = frame.columns.map(str)
    return frame.loc[:, model.features]


def predict(matrix):
    with torch.inference_mode():
        return model(torch.tensor(matrix.values, dtype=torch.float64)).squeeze(-1).numpy()


stage_all = load_stage("center_all")
int(stage_all.isna().sum().sum()), int(stage_all.isna().all(axis=0).sum())

(9000, 375)

In [16]:
diffs = {}
for centering in ["center_all", "center_refgroup"]:
    matrix = load_stage(centering)
    assert matrix.isna().any().any(), "the NaN path is what we are trying to exercise"
    ours = predict(matrix)

    # (1) against the committed ground truth, which is the published pipeline's own output.
    reference = np.array(expected["tagemortality_" + centering])
    diffs[f"tagemortality_{centering} vs expected_predictions.json"] = np.abs(ours - reference).max()

    # (2) against the pipeline re-run here, on the same NaN-bearing matrix. This is the end-to-end
    # proof that reference_values + TAge.preprocess reproduce the fitted SimpleImputer.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        direct = pipeline.predict(matrix) * LIFESPAN_SCALE
    diffs[f"tagemortality_{centering} vs pipeline.predict (NaN path)"] = np.abs(ours - direct).max()

    # (3) NaN-free control: substituting the medians by hand before the forward pass must give the
    # same answer as letting preprocess do it, or the substitution is not what it claims to be.
    filled = matrix.fillna(pd.Series(model.reference_values, index=model.features))
    assert not filled.isna().any().any()
    diffs[f"tagemortality_{centering} preprocess vs hand-filled"] = np.abs(ours - predict(filled)).max()

for label, value in diffs.items():
    print(f"{label:<62} {value:.3e}")

assert max(diffs.values()) < 1e-6, diffs

tagemortality_center_all vs expected_predictions.json          8.882e-16
tagemortality_center_all vs pipeline.predict (NaN path)        8.882e-16
tagemortality_center_all preprocess vs hand-filled             1.110e-15
tagemortality_center_refgroup vs expected_predictions.json     1.166e-15
tagemortality_center_refgroup vs pipeline.predict (NaN path)   1.166e-15
tagemortality_center_refgroup preprocess vs hand-filled        6.661e-16


The recorded unit for this clock, straight from the fixture generator, and a spot check that the
predictions land where the fixtures README says they do:

In [17]:
print(expected["units_tagemortality"])
print("n_features:", expected["n_features_tagemortality"], "==", len(model.features))
assert expected["n_features_tagemortality"] == len(model.features)

ours_all = predict(load_stage("center_all"))
ours_ref = predict(load_stage("center_refgroup"))
pd.DataFrame(
    {"center_all": ours_all, "center_refgroup": ours_ref},
    index=expected["sample_ids"],
).describe()

log10(hazard ratio)
n_features: 10487 == 10487


,center_all,center_refgroup
count,24.000000,24.000000
mean,-0.077299,0.193782
std,0.428031,0.428031
min,-0.668232,-0.397151
25%,-0.410488,-0.139408
50%,-0.146459,0.124622
75%,0.250620,0.521700
max,0.889664,1.160745


## Save torch model

In [18]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

Load it back the way `pyaging` will, and confirm the round trip keeps both the cohort-relative
guard and the numbers. A `.pt` that silently lost `required_uns_flag` would happily score raw
counts.

In [19]:
# weights_only=False is pyaging's load path: a clock is a pickled nn.Module subclass carrying its
# own preprocess, not a bare state dict, so a weights_only load cannot reconstruct it.
reloaded = torch.load(
    f"../weights/{model.metadata['clock_name']}.pt", map_location="cpu", weights_only=False
)
assert reloaded.required_uns_flag == "tage_prepared"
assert reloaded.features == model.features
assert reloaded.reference_values == model.reference_values

reloaded.eval()
reloaded.to(torch.float64)
with torch.inference_mode():
    round_trip = reloaded(torch.tensor(load_stage("center_all").values, dtype=torch.float64)).squeeze(-1).numpy()

round_trip_diff = np.abs(round_trip - ours_all).max()
print("reload max abs diff:", round_trip_diff)
assert round_trip_diff == 0.0

reload max abs diff: 0.0


## Clear directory
<a id="10"></a>

In [20]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: EN_Mortality_Multispecies_Multitissue_scaleddiff.pkl
